# Fuel-Limited Travelling Salesman Problem

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [1]:
def softmax(x):
    # Converts logits into probabilities
    e = np.exp(x - np.max(x))
    return e / e.sum()

def relu(x):
    # Used in hidden layer
    return np.maximum(0, x)

# Calculate L2 Norm
def euclidean(a, b):
    return float(np.linalg.norm(a - b))

# Policy Network

In [2]:
class FuelPolicyNetwork:
    """
    Input  : current city one-hot (20) + visited mask (20) + fuel/max_fuel (1) = 41
    Hidden : 128 units, ReLU
    Output : 20 logits → masked softmax
    """

    def __init__(self, n_cities=20, hidden=128, lr=1e-3):
        self.n = n_cities
        self.lr = lr
        in_dim = 2 * n_cities + 1
        self.W1 = np.random.randn(in_dim, hidden) * np.sqrt(2.0 / in_dim)
        self.b1 = np.zeros(hidden)
        self.W2 = np.random.randn(hidden, n_cities) * np.sqrt(2.0 / hidden)
        self.b2 = np.zeros(n_cities)

    def forward(self, city_idx, visited_mask, fuel_norm, mask_unreachable=None):
        x = np.zeros(2 * self.n + 1)
        x[city_idx] = 1.0
        x[self.n: 2 * self.n] = visited_mask
        x[2 * self.n] = fuel_norm

        h = relu(x @ self.W1 + self.b1)
        logits = h @ self.W2 + self.b2

        # Mask already-visited and unreachable cities
        logits[visited_mask.astype(bool)] = -1e9
        if mask_unreachable is not None:
            logits[mask_unreachable] = -1e9
        probs = softmax(logits)
        return probs, (x, h, logits)

    def select_action(self, city_idx, visited_mask, fuel_norm, mask_unreachable=None):
        probs, cache = self.forward(city_idx, visited_mask, fuel_norm, mask_unreachable)
        action = np.random.choice(self.n, p=probs)
        log_prob = np.log(probs[action] + 1e-9)
        return action, log_prob, cache

    def update(self, episode_data, G):
        dW1 = np.zeros_like(self.W1)
        db1 = np.zeros_like(self.b1)
        dW2 = np.zeros_like(self.W2)
        db2 = np.zeros_like(self.b2)

        for log_prob, (x, h, logits), action in episode_data:
            probs = softmax(logits)
            d_logits = probs.copy()
            d_logits[action] -= 1.0
            d_logits *= -G

            dW2 += np.outer(h, d_logits)
            db2 += d_logits
            dh = d_logits @ self.W2.T
            dh_pre = dh * (h > 0)
            dW1 += np.outer(x, dh_pre)
            db1 += dh_pre

        for grad in [dW1, db1, dW2, db2]:
            np.clip(grad, -5, 5, out=grad)

        k = max(len(episode_data), 1)
        self.W1 -= self.lr * dW1 / k
        self.b1 -= self.lr * db1 / k
        self.W2 -= self.lr * dW2 / k
        self.b2 -= self.lr * db2 / k



# Environment

In [ ]:
class FuelTSPEnv:
    def __init__(self, cities, max_fuel):
        self.cities = cities
        self.n = len(cities)
        self.max_fuel = max_fuel

    def reset(self):
        self.visited = np.zeros(self.n)
        self.current = 0
        self.visited[0] = 1
        self.fuel = self.max_fuel
        self.tour = [0]
        return self.current, self.visited.copy(), self.fuel

    def reachable_mask(self):
        """Boolean array: True = city is unreachable (too far or already visited)."""
        mask = np.zeros(self.n, dtype=bool)
        for j in range(self.n):
            if self.visited[j] or euclidean(self.cities[self.current], self.cities[j]) > self.fuel:
                mask[j] = True
        return mask

    def step(self, action):
        dist = euclidean(self.cities[self.current], self.cities[action])
        self.fuel -= dist
        self.visited[action] = 1
        self.tour.append(action)
        self.current = action

        reward = 1.0   # reward for visiting a new city
        # Check done
        unreach = self.reachable_mask()
        all_unreachable = unreach.sum() == self.n   # all masked = visited or can't reach
        done = all_unreachable or int(self.visited.sum()) == self.n
        return self.current, self.visited.copy(), self.fuel, reward, done


# Training Loop

In [ ]:
def generate_cities(n=20, seed=42):
    rng = np.random.RandomState(seed)
    return rng.rand(n, 2)

def run_episode(policy, env, max_fuel):
    current, visited, fuel = env.reset()
    episode_data = []
    total_reward = 0.0

    for _ in range(env.n):
        fuel_norm = fuel / max_fuel
        unreach_mask = env.reachable_mask()

        # If no city reachable, stop
        if unreach_mask.all():
            break

        action, log_prob, cache = policy.select_action(
            current, visited, fuel_norm, unreach_mask)

        next_city, visited, fuel, reward, done = env.step(action)
        episode_data.append((log_prob, cache, action))
        total_reward += reward
        current = next_city
        if done:
            break

    return total_reward, episode_data, env.tour[:]

def greedy_run(policy, env, max_fuel):
    current, visited, fuel = env.reset()
    for _ in range(env.n):
        fuel_norm = fuel / max_fuel
        unreach_mask = env.reachable_mask()
        if unreach_mask.all():
            break
        probs, _ = policy.forward(current, visited, fuel_norm, unreach_mask)
        action = int(np.argmax(probs))
        current, visited, fuel, reward, done = env.step(action)
        if done:
            break
    return len(env.tour), env.tour[:]

def train_fuel_tsp(max_fuel, n_episodes=5000, n_cities=20, lr=1e-3, baseline_decay=0.99):
    cities = generate_cities(n_cities)
    env = FuelTSPEnv(cities, max_fuel)
    policy = FuelPolicyNetwork(n_cities, hidden=128, lr=lr)

    rewards_history = []
    baseline = None
    best_tour = None
    best_cities_visited = 0

    for ep in range(n_episodes):
        reward, episode_data, tour = run_episode(policy, env, max_fuel)

        if not episode_data:
            continue

        if baseline is None:
            baseline = reward
        else:
            baseline = baseline_decay * baseline + (1 - baseline_decay) * reward

        G = reward - baseline
        policy.update(episode_data, G)
        rewards_history.append(reward)

        if reward > best_cities_visited:
            best_cities_visited = reward
            best_tour = tour[:]

    final_count, final_tour = greedy_run(policy, env, max_fuel)
    return policy, cities, final_tour, final_count, rewards_history


# Fuel Variation Study

In [3]:
def fuel_variation_study():
    n_cities = 20
    cities = generate_cities(n_cities)
    # Estimate max possible tour distance
    max_possible = np.sqrt(2) * n_cities   # very rough upper bound

    fuel_levels = np.linspace(0.5, 6.0, 12)
    cities_visited_results = []

    print("\nQ3 Fuel Variation Study:")
    print(f"{'Fuel':>8} | {'Cities Visited (greedy)':>22}")
    print("-" * 35)

    for fuel in fuel_levels:
        _, cities_data, tour, count, hist = train_fuel_tsp(
            max_fuel=fuel, n_episodes=3000, n_cities=n_cities)
        cities_visited_results.append(count)
        print(f"  {fuel:6.2f} | {count:>22d}")

    # Plot fuel vs cities visited
    plt.figure(figsize=(8, 5))
    plt.plot(fuel_levels, cities_visited_results, 'bo-', linewidth=2, markersize=8)
    plt.xlabel("Fuel Capacity", fontsize=12)
    plt.ylabel("Cities Visited", fontsize=12)
    plt.title("Q3: Effect of Fuel Capacity on Cities Visited", fontsize=13)
    plt.axhline(y=n_cities, color='r', linestyle='--', label=f'All {n_cities} cities')
    plt.legend(); plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig("q3_fuel_variation.png", dpi=100)
    plt.close()
    return fuel_levels, cities_visited_results

def plot_tour_with_fuel(cities, tour, title, fname, max_fuel):
    fig, ax = plt.subplots(figsize=(7, 7))
    n = len(tour)
    colors = cm.viridis(np.linspace(0, 1, n))
    for i in range(n - 1):
        a, b = cities[tour[i]], cities[tour[i + 1]]
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="->", color=colors[i], lw=2))
    for i, (x, y) in enumerate(cities):
        visited = i in tour
        ax.plot(x, y, 'go' if visited else 'rx', markersize=10 if visited else 8)
        ax.text(x + 0.01, y + 0.01, str(i), fontsize=8)
    ax.set_title(f"{title}\nVisited {len(tour)} / {len(cities)} cities | Fuel={max_fuel:.1f}")
    ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
    from matplotlib.lines import Line2D
    legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor='g',
                               markersize=10, label='Visited'),
                       Line2D([0], [0], marker='x', color='r', markersize=10, label='Skipped')]
    ax.legend(handles=legend_elements)
    plt.tight_layout()
    plt.savefig(fname, dpi=100)
    plt.close()

In [ ]:
# Solve with a moderate fuel level
fuel = 3.0
print(f"Q3: Fuel-Limited TSP (fuel={fuel})")
policy, cities, tour, count, history = train_fuel_tsp(max_fuel=fuel, n_episodes=5000)

print(f"\n  Cities visited (greedy): {count} / {le(cities)}")
print(f"  Tour: {tour}")

plot_tour_with_fuel(cities, tour, "Q3 Fuel-Limited TSP", "q3_tour.png", fuel)

# Training curve
window = 200
smoothed = np.convolve(history, np.ones(window) / window, mode='valid')
plt.figure(figsize=(8, 4))
plt.plot(history, alpha=0.2, color='blue')
plt.plot(range(window - 1, len(history)), smoothed, color='red', label=f'{window}-ep avg')
plt.xlabel("Episode"); plt.ylabel("Cities Visited (sampled)")
plt.title(f"Q3 Fuel-Limited TSP Training (fuel={fuel})")
plt.legend(); plt.tight_layout()
plt.savefig("q3_training.png", dpi=100)
plt.close()

# Fuel variation study
fuel_levels, cities_visited = fuel_variation_study()
print("\nSaved: q3_tour.png, q3_training.png, q3_fuel_variation.png")